# Chapter Title Generator
**Problem Statement:** Authors often struggle with coming up with a creative, innovative, and/or fitting title for the chapters of their book or even the book itself. This process can interrupt writing flow and may be particularly difficult for new writers. 

**Proposed Methodology:** We propose a supervised deep learning approach using transformer-based sequence-to-sequence text generation models to solve this problem. We plan to train a model to generate a list of suitable titles


In [1]:
# Imports
import os
import pandas as pd
from datasets import load_dataset
import re
from transformers import BertTokenizerFast
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import EncoderDecoderModel
import transformers
import accelerate
from transformers import TrainerCallback
import gc
from transformers import DataCollatorForSeq2Seq
import torch

In [2]:
# Data Loading
stories_dataset = load_dataset("FareedKhan/1k_stories_100_genre", split="train")
cmu_dataset = load_dataset("textminr/cmu-book-summaries", split="train")
'''
if not os.path.exists("novel-chapter-dataset"):
    !git clone https://github.com/manestay/novel-chapter-dataset.git
else:
    print("Repository already cloned.")
'''
print(stories_dataset.column_names)
print(cmu_dataset.column_names)

['id', 'title', 'story', 'genre']
['title', 'author', 'pub_year', 'summary']


In [7]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Basic profanity filter using regex
def filter_profanity(text):
    bad_words_pattern = re.compile(r'\b(shit|fuck|bitch|cunt|slut|dick)\b', re.IGNORECASE)
    return not bool(bad_words_pattern.search(text))

# Instruction tuning formatter and chunker
def preprocess_function(examples):
    # Instruction tuning prompt
    inputs = ["Generate a chapter title for the following text: " + doc for doc in examples["story"]]
    
    # Tokenize and chunk the input texts
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")
    
    # Tokenize the target titles
    labels = tokenizer(examples["title"], max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# Apply functions to dataset
clean_dataset = stories_dataset.filter(lambda x: filter_profanity(x['story']) and filter_profanity(x['title']))
tokenized_datasets = clean_dataset.map(preprocess_function, batched=True, remove_columns=clean_dataset.column_names)
split_datasets = tokenized_datasets.train_test_split(test_size=0.2)

In [8]:
unified_data = []

for row in stories_dataset:
    unified_data.append({"input_text": row["story"], "target_title": row["title"]})

# Convert to a unified Pandas DataFrame, then back to a Hugging Face Dataset
df = pd.DataFrame(unified_data)
# Drop any rows that have missing values
df = df.dropna().reset_index(drop=True)

print(f"Total unified training pairs: {len(df)}")

Total unified training pairs: 1000


In [9]:
model = EncoderDecoderModel.from_encoder_decoder_pretrained("bert-base-uncased", "bert-base-uncased")

# Crucial configuration for BERT-based decoding
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.weight                                        | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
bert.pooler.dense.bias                                             | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.query.bias         | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.dense.weight     | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
bert.encoder.layer.{

In [12]:
import torch
import gc

# Force garbage collection and empty the MPS cache
gc.collect()
torch.mps.empty_cache()
print("MPS Memory Cache Cleared!")

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./title_generator_model",
    eval_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=1,
    save_strategy="epoch",
    num_train_epochs=1,
    predict_with_generate=False,
    fp16=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    data_collator=data_collator,
    processing_class=tokenizer,
)
model.gradient_checkpointing_enable()
# Launch the fine-tuning process
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None}.


MPS Memory Cache Cleared!


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=40.2353662109375, metrics={'train_runtime': 1205.6302, 'train_samples_per_second': 0.661, 'train_steps_per_second': 0.041, 'total_flos': 244461459041280.0, 'train_loss': 40.2353662109375, 'epoch': 1.0})

In [19]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def generate_creative_titles(story_text, num_options=3):
    prompt = "Generate a chapter title for the following text: " + story_text
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    decoder_start_id = tokenizer.cls_token_id
    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.sep_token_id
    
    outputs = model.generate(
        inputs.input_ids,
        decoder_start_token_id=decoder_start_id,
        bos_token_id=decoder_start_id,
        pad_token_id=pad_id,
        eos_token_id=eos_id,
        
        max_length=15,               # Titles should be short
        
        do_sample=True,            
        temperature=0.8,           
        top_k=40,                    
        top_p=0.9,                  
        
        repetition_penalty=2.5,    
        no_repeat_ngram_size=1,      # Absolutely forbids repeating any single word within a title
        
        num_return_sequences=num_options,
    )
    
    generated_titles = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]
    
    # Quick cleaning step: strip out loose commas/dots if the decoder gets lazy
    cleaned_titles = [title.strip(" .,;:-") for title in generated_titles]
    return cleaned_titles

In [20]:
# Cell: Testing with custom creative text

sample_story_1 = """
The atmospheric regulators hissed as Leo adjusted his oxygen mask. 
Outside the viewport, the crimson sands of the new colony were being swallowed by a massive dust storm. 
He checked the telemetry data one last time—if the shield generator failed tonight, the entire dome would depressurize.
"""

print("--- Generating Titles for Sample 1 ---")
suggested_titles_1 = generate_creative_titles(sample_story_1, num_options=3)

for i, title in enumerate(suggested_titles_1, 1):
    print(f"Option {i}: {title}")

--- Generating Titles for Sample 1 ---
Option 1: of
Option 2: the of
Option 3: 
